### 1. Clone RCS-YOLO Repository and Change Directory

In [1]:
# Clone the RCS-YOLO repository from GitHub
!git clone https://github.com/mkang315/RCS-YOLO.git

Cloning into 'RCS-YOLO'...
Updating files:  96% (1410/1457)
Updating files:  97% (1414/1457)
Updating files:  98% (1428/1457)
Updating files:  99% (1443/1457)
Updating files: 100% (1457/1457)
Updating files: 100% (1457/1457), done.


* Change Folder name

In [ ]:
import os
os.rename("brain tumor detection.v2-mahitha.yolov8", "brain_tumor_detection")

* Move Folder to other directory

In [1]:
!move brain_tumor_detection RCS-YOLO

        1개의 디렉터리를 이동했습니다.


In [1]:
# Change the current working directory into the cloned repository folder
%cd RCS-YOLO

C:\Users\wm473\Downloads\Week05\RCS-YOLO


D:\ANA3\envs\ComputerVision\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


### 2. Dataset Class, DataLoader Definition

In [2]:
import os
import glob
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
# import yaml # PyYAML installation needed if you parse data.yaml: pip install PyYAML
import re # Added for using regular expressions

# --- 1. Custom Dataset Class (with improved name matching logic) ---
class YOLOv8Dataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform

        # Get list of image files (including case-insensitive extensions)
        img_patterns = ['*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG', '*.JPEG']
        self.img_files = []
        for pattern in img_patterns:
            self.img_files.extend(glob.glob(os.path.join(img_dir, pattern)))
        self.img_files = sorted(list(set(self.img_files))) # Remove duplicates and sort

        print(f"Debug: Found {len(self.img_files)} potential image files in {img_dir}")
        if not self.img_files:
             print(f"Debug: No image files found in {img_dir} with patterns {img_patterns}")
             self.label_map = {}
             self.valid_img_files = []
             self.valid_label_files = []
             return # No need to proceed if no images found

        # Extract base names and paths for label files (considering Roboflow format)
        self.label_map = {} # Maps image core_name to label path
        raw_label_files = glob.glob(os.path.join(label_dir, '*.txt'))
        print(f"Debug: Found {len(raw_label_files)} potential label files in {label_dir}")

        for label_path in raw_label_files:
            base_name_with_ext = os.path.basename(label_path)
            base_name = os.path.splitext(base_name_with_ext)[0] # Remove .txt extension
            # Attempt to remove Roboflow hash (e.g., .rf.xxxxxxxx...)
            core_name = re.sub(r'\.rf\.[a-f0-9]+$', '', base_name)
            # Attempt to remove original extension if included in the name (e.g., _jpg)
            core_name = re.sub(r'(_jpg|_png|_jpeg)$', '', core_name, flags=re.IGNORECASE)

            self.label_map[core_name] = label_path # Use core_name as the key for matching

        if not self.label_map:
             print(f"Debug: No label files found or processed in {label_dir}")

        # Match image files with label files (considering Roboflow format)
        self.valid_img_files = []
        self.valid_label_files = [] # Store paths of matched labels
        matched_count = 0
        unmatched_img_examples = []

        for img_path in self.img_files:
            img_base_name_with_ext = os.path.basename(img_path)
            img_base_name = os.path.splitext(img_base_name_with_ext)[0] # Remove extension

            # Attempt to remove hash and potential included extension parts from image name
            img_core_name = re.sub(r'\.rf\.[a-f0-9]+$', '', img_base_name)
            img_core_name = re.sub(r'(_jpg|_png|_jpeg)$', '', img_core_name, flags=re.IGNORECASE)

            # Find the matching core_name in the label map
            if img_core_name in self.label_map:
                self.valid_img_files.append(img_path)
                self.valid_label_files.append(self.label_map[img_core_name]) # Add matched label path
                matched_count += 1
            else:
                if len(unmatched_img_examples) < 5: # Store up to 5 examples of unmatched images
                    unmatched_img_examples.append((img_path, img_core_name))

        print(f"Debug: Matched {matched_count} image-label pairs.")

        if not self.valid_img_files:
             print(f"Warning: Could not find matching label files for images. Check file naming conventions.")
             # Print examples for debugging
             if unmatched_img_examples:
                 print("  Unmatched image examples (path, derived core_name):")
                 for img_ex_path, img_ex_core in unmatched_img_examples:
                      print(f"    - {img_ex_path}, '{img_ex_core}'")
             if self.label_map:
                 example_label_key = list(self.label_map.keys())[0]
                 print(f"  Example label core_name derived: '{example_label_key}' (from '{self.label_map[example_label_key]}')")

        # self.img_files now only contains valid files with matched labels
        self.img_files = self.valid_img_files

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        if idx >= len(self.img_files):
             raise IndexError("Index out of range")

        img_path = self.img_files[idx]
        # Use the label file path that was already matched in __init__
        label_path = self.valid_label_files[idx]

        # Load image (convert to RGB)
        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            print(f"Error: Failed to load image '{img_path}': {e}")
            return None # To be handled by collate_fn

        # Load labels
        labels = []
        if label_path and os.path.exists(label_path):
            try:
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            try:
                                # Class index and coordinates (convert all to float)
                                class_idx = float(parts[0])
                                coords = [float(p) for p in parts[1:]]
                                labels.append([class_idx] + coords)
                            except ValueError:
                                print(f"Warning: Error converting label line to numbers in '{label_path}': '{line.strip()}'")
                        # else: # Can produce too many warnings
                        #     print(f"Warning: Incorrect format in label file '{label_path}': '{line.strip()}'")
            except Exception as e:
                print(f"Error: Failed to read label file '{label_path}': {e}")
                labels = []

        # Convert list to NumPy array then to Tensor
        # shape: (num_objects, 5)
        labels_tensor = torch.tensor(labels, dtype=torch.float32)
        if not labels: # If label file was empty or had errors
             labels_tensor = torch.empty((0, 5), dtype=torch.float32)

        # Apply image transformations (transform)
        if self.transform:
            try:
                image = self.transform(image)
            except Exception as e:
                 print(f"Error: Failed to transform image '{img_path}': {e}")
                 return None # To be handled by collate_fn

        # Final format to be passed to DataLoader (tuple or dictionary)
        return image, labels_tensor

# --- 2. Collate Function for DataLoader (Same as before) ---
def yolo_collate_fn(batch):
    # Filter out samples where __getitem__ returned None
    batch = [item for item in batch if item is not None]
    if not batch:
        # If all items were None, return empty batch or raise exception
        print("Warning: No valid samples to process in collate_fn.")
        return None, None # Or return torch.empty(0), []

    try:
        # Stack images into a batch tensor
        images = torch.stack([item[0] for item in batch], 0)
        # Keep labels as a list (because they can have different numbers of objects)
        labels = [item[1] for item in batch]
        return images, labels
    except RuntimeError as e:
        # Error likely occurs if image tensors in the batch have different shapes
        print(f"Error: Runtime error during batch collation: {e}")
        print("Individual image tensor shapes:")
        for i, item in enumerate(batch):
            if hasattr(item[0], 'shape'):
                print(f"  Item {i} image shape: {item[0].shape}")
            else:
                print(f"  Item {i} image is not a tensor (type: {type(item[0])})")
        # Return None or raise exception if problem persists
        return None, None


# --- 3. Usage Example (Paths corrected) ---
if __name__ == '__main__':
    # --- Specify paths directly (Corrected) ---
    # Use raw strings (r"...") or escape backslashes ("\\") for Windows paths
    # Add the "brain_tumor_detection" folder to the path.
    base_data_dir = r"C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection" # <--- Root folder of the dataset

    train_img_dir = os.path.join(base_data_dir, "train", "images")
    train_label_dir = os.path.join(base_data_dir, "train", "labels")
    val_img_dir = os.path.join(base_data_dir, "valid", "images")
    val_label_dir = os.path.join(base_data_dir, "valid", "labels")
    # Add test paths similarly if needed
    # test_img_dir = os.path.join(base_data_dir, "test", "images")
    # test_label_dir = os.path.join(base_data_dir, "test", "labels")
    # ---------------------

    print(f"Train image directory: {train_img_dir}")
    print(f"Train label directory: {train_label_dir}")
    print(f"Validation image directory: {val_img_dir}")
    print(f"Validation label directory: {val_label_dir}")

    # Check if directories exist
    if not os.path.isdir(train_img_dir):
        print(f"Error: Train image directory not found: {train_img_dir}")
        exit()
    if not os.path.isdir(train_label_dir):
        print(f"Error: Train label directory not found: {train_label_dir}")
        exit()
    if not os.path.isdir(val_img_dir):
        print(f"Warning: Validation image directory not found: {val_img_dir}")
    if not os.path.isdir(val_label_dir):
        print(f"Warning: Validation label directory not found: {val_label_dir}")

    # Define image transformations
    transform = T.Compose([
        T.Resize((640, 640)), # Typical input size for YOLOv8
        T.ToTensor(),
        # T.Normalize(mean=[...], std=[...]) # Add normalization if needed
    ])

    # Create Train Dataset and DataLoader
    try:
        train_dataset = YOLOv8Dataset(img_dir=train_img_dir,
                                      label_dir=train_label_dir,
                                      transform=transform)

        if len(train_dataset) == 0:
             print("Error: No valid samples found after creating Train dataset. Check Debug messages in __init__.")
             # If paths are correct but count is 0, likely a file name matching issue
             # Ensure the Dataset class edits from previous answer were applied
             exit()

        train_loader = DataLoader(train_dataset,
                                  batch_size=4,      # Reduced batch size for testing
                                  shuffle=True,      # Shuffle training data
                                  num_workers=0,     # Set to 0 for debugging, especially on Windows
                                  collate_fn=yolo_collate_fn, # Use custom collate function
                                  pin_memory=True)   # Optimize data transfer to GPU if available

        print(f"\nTrain dataset loaded successfully. Total samples: {len(train_dataset)}")

        # Test loading a batch from the DataLoader
        for i, batch_data in enumerate(train_loader):
             if batch_data is None or batch_data[0] is None:
                 print(f"--- Failed to load Batch {i+1} (collate_fn returned None) ---")
                 continue # Try next batch

             images, labels = batch_data
             print(f"\n--- Batch {i+1} ---")
             print(f"Images batch shape: {images.shape}") # Expected: [batch_size, C, H, W]
             print(f"Labels batch length: {len(labels)}") # Expected: batch_size (list of tensors)
             for j, label_tensor in enumerate(labels):
                 print(f"  Label tensor {j} shape: {label_tensor.shape}") # Expected: [Num_objects_in_img_j, 5]
             if i == 0:
                 break # Only check the first batch

    except Exception as e:
        print(f"Error during Train data loading or DataLoader creation: {e}")
        import traceback
        traceback.print_exc() # Print detailed error stack


    # Create Validation Dataset and DataLoader
    try:
        # Only create validation loader if directories exist
        if os.path.isdir(val_img_dir) and os.path.isdir(val_label_dir):
            val_dataset = YOLOv8Dataset(img_dir=val_img_dir,
                                        label_dir=val_label_dir,
                                        transform=transform)

            if len(val_dataset) > 0:
                val_loader = DataLoader(val_dataset,
                                        batch_size=4,
                                        shuffle=False, # No need to shuffle validation data
                                        num_workers=0,
                                        collate_fn=yolo_collate_fn,
                                        pin_memory=True)
                print(f"\nValidation dataset loaded successfully. Total samples: {len(val_dataset)}")
            else:
                 print("Warning: No valid samples found after creating Validation dataset.")
        else:
            print("Warning: Validation directories not valid. Skipping Validation DataLoader creation.")

    except Exception as e:
        print(f"Error during Validation data loading: {e}")
        import traceback
        traceback.print_exc()

Train image directory: C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\train\images
Train label directory: C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\train\labels
Validation image directory: C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\valid\images
Validation label directory: C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\valid\labels
Debug: Found 397 potential image files in C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\train\images
Debug: Found 397 potential label files in C:\Users\wm473\Downloads\Week05\RCS-YOLO\brain_tumor_detection\train\labels
Debug: Matched 397 image-label pairs.

Train dataset loaded successfully. Total samples: 397

--- Batch 1 ---
Images batch shape: torch.Size([4, 3, 640, 640])
Labels batch length: 4
  Label tensor 0 shape: torch.Size([1, 5])
  Label tensor 1 shape: torch.Size([1, 5])
  Label tensor 2 shape: torch.Size([1, 5])
  Label tensor 3 shape: torch.Size([1, 5])
Debu

### 3. Load Model & Forward pass

In [9]:
import torch
import yaml
from pathlib import Path

# --- 1. Manually Set Command-line Arguments ---
class DummyOpt:
    pass

opt = DummyOpt()
opt.workers = 8
opt.device = "cpu"  # Set to "cpu" if GPU is not available
opt.batch_size = 32
opt.data = "data/br35h.yaml"         # Data configuration file (includes number of classes, names, etc.)
opt.img_size = [640, 640]              # Image sizes for [train, test]
opt.cfg = "cfg/training/rcs-yolo.yaml" # Model configuration file
opt.weights = ""                     # If empty, create model from scratch (no pretrained weights)
opt.name = "rcs-yolo"
opt.hyp = "data/hyp_training.yaml"   # Hyperparameter file
opt.single_cls = False               # Whether to treat data as a single class

# --- 2. Load Device, Hyperparameters, and Dataset Information ---
device = torch.device("cuda:" + opt.device if torch.cuda.is_available() else "cpu")
print("Device:", device)

with open(opt.hyp, 'r') as f:
    hyp = yaml.safe_load(f)
print("Hyperparameters loaded from", opt.hyp)

with open(opt.data, 'r') as f:
    data_dict = yaml.safe_load(f)
nc = 1 if opt.single_cls else int(data_dict['nc'])
names = data_dict['names']
print(f"Number of classes: {nc}, names: {names}")

# --- 3. Load Model (Using Model class from models/yolo.py) ---
from models.yolo import Model
model = Model(opt.cfg, ch=3, nc=nc, anchors=hyp.get('anchors')).to(device)
model.train()  # Set model to training mode
print("Model loaded from", opt.cfg)

# Assign additional attributes required by ComputeLoss (e.g., hyp and gr)
model.hyp = hyp
model.gr = 1.0   # IOU loss ratio (example value)

# --- 4. Get a Batch from the DataLoader ---
# Assume that a custom DataLoader (train_loader) has already been created.
batch = next(iter(train_loader))
images, labels = batch
print("\nUsing batch from train_loader:")
print(f"Images batch shape: {images.shape}")
print(f"Number of label tensors: {len(labels)}")
for j, label_tensor in enumerate(labels):
    print(f"  Label tensor {j} shape: {label_tensor.shape}")

# --- 5. Construct Targets Tensor ---
# For YOLO loss, each label needs to include the image index.
all_targets = []
for i, lab in enumerate(labels):
    if lab.shape[0] > 0:
        image_idx = torch.full((lab.shape[0], 1), i, dtype=lab.dtype, device=lab.device)
        all_targets.append(torch.cat((image_idx, lab), dim=1))
if all_targets:
    targets_tensor = torch.cat(all_targets, dim=0)  # Shape: [total_objects, 6]
else:
    targets_tensor = torch.empty((0, 6), device=device)
print(f"Constructed targets tensor of shape: {targets_tensor.shape}")

# --- 6. Run Model Forward Pass ---
outputs = model(images)
print("\n--- Forward Pass ---")
if isinstance(outputs, (list, tuple)):
    for idx, output in enumerate(outputs):
        print(f"Output {idx} shape: {output.shape}, sum: {output.sum().item()}")
else:
    print("Output shape:", outputs.shape, "sum:", outputs.sum().item())

# --- 7. Compute Loss (Using ComputeLoss) ---
from utils.loss import ComputeLoss
compute_loss = ComputeLoss(model)  # ComputeLoss is defined in utils/loss.py
loss, loss_items = compute_loss(outputs, targets_tensor)
# loss_items: [box_loss, obj_loss, cls_loss, total_loss]
print("\nLoss components:")
print(f"  Box Loss: {loss_items[0].item()}")
print(f"  Obj Loss: {loss_items[1].item()}")
print(f"  Cls Loss: {loss_items[2].item()}")
print(f"  Total Loss: {loss_items[3].item()}")

# --- 8. Run Backward Pass ---
loss.backward()
print("\nBackward pass completed. Some gradient values:")
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        print(f"  {name} gradient norm: {grad_norm}")
        break

print("\nForward and backward pass with ComputeLoss completed successfully.")

Device: cpu
Hyperparameters loaded from data/hyp_training.yaml
Number of classes: 1, names: ['Brain Tumor']
Model loaded from cfg/training/rcs-yolo.yaml

Using batch from train_loader:
Images batch shape: torch.Size([4, 3, 640, 640])
Number of label tensors: 4
  Label tensor 0 shape: torch.Size([2, 5])
  Label tensor 1 shape: torch.Size([2, 5])
  Label tensor 2 shape: torch.Size([1, 5])
  Label tensor 3 shape: torch.Size([2, 5])
Constructed targets tensor of shape: torch.Size([7, 6])

--- Forward Pass ---
Output 0 shape: torch.Size([4, 2, 40, 40, 6]), sum: -12890.201171875
Output 1 shape: torch.Size([4, 2, 20, 20, 6]), sum: 4410.5654296875

Loss components:
  Box Loss: 0.07849567383527756
  Obj Loss: 0.11806000769138336
  Cls Loss: 0.0
  Total Loss: 0.19655567407608032

Backward pass completed. Some gradient values:
  model.0.rbr_dense.conv.weight gradient norm: 3.8647091388702393

Forward and backward pass with ComputeLoss completed successfully.
